# Exploring Network Architecture

The previous experiments introduced the basic computational components of a
feed-forward neural network and demonstrated how activation functions influence
learning.

This experiment investigates a different question:

> How does the architecture of a neural network affect its ability to learn?

A network's architecture determines how information flows through the model.
Changing the number of hidden layers or the number of neurons per layer changes
the model's representational capacity and the number of trainable parameters.

To isolate the effect of architecture, every experiment in this notebook uses:

- the same training data,
- the same activation function,
- the same loss function,
- the same optimizer.

Only the network architecture changes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nnlab.kernels import LogisticKernel

from nnlab.activations import ParameterizedActivation

from nnlab.layers import (
    Dense,
    ActivationLayer,
)

from nnlab.models import FeedForward

from nnlab.losses import MeanSquaredError

from nnlab.optimizers import SGD

from nnlab.training import Trainer

## Create Training Data

A smooth nonlinear function is used as the learning target.

Unlike the previous examples, this function requires the network to learn both
oscillatory and linear behavior.

Every architecture will be trained using exactly the same dataset.

In [ ]:
x = np.linspace( -3.0, 3.0, 300, ).reshape( -1, 1, )

target = ( np.sin(2 * x) + 0.25 * x )

In [ ]:
plt.figure(figsize=(8,4))

plt.plot(
    x,
    target,
)

plt.title(
    "Training Target"
)

plt.xlabel(
    "Input"
)

plt.ylabel(
    "Output"
)

plt.grid(True)

plt.show()

## Build Network Architectures

Each model uses the same activation function and output layer.

Only the number of hidden layers and hidden neurons changes.

In [ ]:
activation = ParameterizedActivation(
    kernel=LogisticKernel(),
)

In [ ]:
def build_model(
    hidden_layers,
    hidden_units,
):

    layers = []

    input_size = 1

    for _ in range(hidden_layers):

        layers.append(
            Dense(
                input_size=input_size,
                output_size=hidden_units,
            )
        )

        layers.append(
            ActivationLayer(
                ParameterizedActivation(
                    kernel=LogisticKernel(),
                )
            )
        )

        input_size = hidden_units

    layers.append(
        Dense(
            input_size=input_size,
            output_size=1,
        )
    )

    return FeedForward(
        layers=layers,
    )

## Compare Architectures

The following networks will be trained.

The notation

depth × width

indicates:

- depth = hidden layers
- width = neurons in each hidden layer

In [ ]:
architectures = {

    "1×8": (1,8),

    "1×32": (1,32),

    "2×16": (2,16),

    "3×16": (3,16),

    "4×16": (4,16),

}

## Train Each Network

Each architecture is trained using:

- Mean Squared Error
- Stochastic Gradient Descent
- identical learning rate
- identical number of epochs

This isolates architecture as the only experimental variable.

In [ ]:
results = {}

for name, (depth, width) in architectures.items():

    model = build_model(
        hidden_layers=depth,
        hidden_units=width,
    )

    trainer = Trainer(
        model=model,
        loss=MeanSquaredError(),
        optimizer=SGD(
            learning_rate=0.025,
        ),
    )

    history = trainer.fit(
        x,
        target,
        epochs=2500,
    )

    results[name] = {

        "model": model,

        "history": history,

    }

## Compare Training Loss

The loss history illustrates how quickly each architecture converges during
optimization.

In [ ]:
plt.figure(figsize=(8, 4))

for name, result in results.items():

    plt.plot(
        result["history"],
        label=name,
    )

plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error")
plt.yscale("log")
plt.legend()
plt.grid(True)

plt.show()

## Compare Learned Functions

Although every model is trained on the same dataset, differences in
architecture influence the quality of the final approximation.

The target function is shown using a dashed black line.

In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(
    x,
    target,
    "k--",
    linewidth=3,
    label="Target",
)

for name, result in results.items():

    prediction = result["model"].forward(x)

    plt.plot(
        x,
        prediction,
        label=name,
    )

plt.title("Learned Function by Network Architecture")
plt.xlabel("Input")
plt.ylabel("Output")
plt.legend()
plt.grid(True)

plt.show()

## Compare Model Complexity

Increasing network depth and width increases the number of trainable
parameters.

Additional parameters provide greater representational capacity but also
increase computational cost.

In [ ]:
def parameter_count(
    model,
):

    return sum(
        parameter.size
        for parameter in model.parameters()
    )


for name, result in results.items():

    print(
        f"{name:6s} : "
        f"{parameter_count(result['model'])} parameters"
    )

## Observations

Network architecture controls the expressive power of a neural network.

Increasing the number of hidden layers or hidden neurons increases the model's
capacity to represent complex functions, but also increases the number of
trainable parameters.

No single architecture is universally best. The appropriate architecture
depends on the complexity of the learning problem and the computational
resources available.

In the next experiment we will investigate how different optimization
algorithms influence the learning process while keeping the network
architecture fixed.